In [ ]:

%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/PixelGen/')

import anndata as ad
import pixelator
import torch
import scvi
import scipy

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import tempfile

scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents,get_dense,calculate_metrics,plot_composite_ppc
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
MODEL_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/models'
SEED = 30
import random

np.random.seed(SEED)
random.seed(SEED)


In [ ]:
adata=sc.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/cache/pbmsc_adata_final_annotated.h5ad')


In [ ]:
cd8= adata[adata.obs['cell_type']=='CD8'    ].copy()
cd8

In [ ]:
transformed_matrix = np.arcsinh(adata.obsm['HOTSPOT_top500_var'] * 100)
adata.obsm['hotspot_500_arcsinh'] = transformed_matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd

# --- 1. ANALYSIS & PREPARE DATA ---
# Recalculate rank genes to ensure 'names' are correct for the dotplot/table
sc.tl.rank_genes_groups(adata, groupby='condition', method='wilcoxon', layer='arcsinh')
result = sc.get.rank_genes_groups_df(adata, group='PHA').copy()

# Add metrics for plotting
result['nlog10'] = -np.log10(result['pvals_adj'] + 1e-300)
fc_thresh = 1.0; pval_thresh = 0.05
result['color'] = np.where((result.pvals_adj < pval_thresh) & (result.logfoldchanges > fc_thresh), 'Up in PHA',
                  np.where((result.pvals_adj < pval_thresh) & (result.logfoldchanges < -fc_thresh), 'Up in Unstim', 'NS'))

# --- 2. FIGURES ---
# A. Volcano Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=result, x='logfoldchanges', y='nlog10', hue='color', 
                palette={'Up in PHA':'#d62728', 'Up in Unstim':'#1f77b4', 'NS':'lightgrey'}, alpha=0.8)

# Label top 5
top_genes = pd.concat([result.nlargest(5, 'logfoldchanges'), result.nsmallest(5, 'logfoldchanges')])
for _, row in top_genes.iterrows():
    plt.text(row['logfoldchanges'], row['nlog10'], row['names'], fontsize=9, 
             ha='right' if row['logfoldchanges'] < 0 else 'left')

plt.axhline(-np.log10(pval_thresh), c='grey', ls='--'); plt.axvline(fc_thresh, c='grey', ls='--'); plt.axvline(-fc_thresh, c='grey', ls='--')
plt.title('Volcano Plot: PHA vs Unstim (Abundance)'); sns.despine(); plt.show()

# B. Heatmap
top_genes_list = result.nlargest(5, 'logfoldchanges')['names'].tolist() + result.nsmallest(5, 'logfoldchanges')['names'].tolist()
sc.pl.heatmap(adata, var_names=top_genes_list, groupby='condition', use_raw=False, layer='arcsinh', 
              standard_scale='var', cmap='RdBu_r', vmin=-2, vmax=2, figsize=(6, 6), swap_axes=True)

# --- 3. TOP 20 ABUNDANCE TABLE ---
top20_list = []
for group in adata.obs['condition'].unique():
    df = sc.get.rank_genes_groups_df(adata, group=group).head(20)
    df['group'] = group
    top20_list.append(df[['group', 'names', 'logfoldchanges', 'pvals_adj']])

print("Top 20 Abundance Markers per Group:")
display(pd.concat(top20_list).reset_index(drop=True))

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Setup & Analysis
# CORRECTED: Use the arcsinh transformed data we created earlier
spatial_adata = ad.AnnData(X=adata.obsm['HOTSPOT_top500_var'], obs=adata.obs)
spatial_adata.var_names = adata.obsm['HOTSPOT_top500_var'].columns
sc.tl.rank_genes_groups(spatial_adata, groupby='condition', method='wilcoxon')

# 2. Figures
# A. Volcano Plot (PHA vs Unstim)
res = sc.get.rank_genes_groups_df(spatial_adata, group='PHA').copy()
res['nlog10'] = -np.log10(res['pvals_adj'] + 1e-300)
res['color'] = np.where((res.pvals_adj<.05) & (res.logfoldchanges>1), 'PHA', 
               np.where((res.pvals_adj<.05) & (res.logfoldchanges<-1), 'Unstim', 'NS'))

plt.figure(figsize=(10, 8))
sns.scatterplot(data=res, x='logfoldchanges', y='nlog10', hue='color', 
                palette={'PHA':'#d62728','Unstim':'#1f77b4','NS':'lightgrey'}, alpha=0.8)

# Add text labels for top 5 hits on each side
top_hits = pd.concat([res.nlargest(5, 'logfoldchanges'), res.nsmallest(5, 'logfoldchanges')])
for _, row in top_hits.iterrows():
    plt.text(row['logfoldchanges'], row['nlog10'], row['names'], fontsize=9, 
             ha='right' if row['logfoldchanges'] < 0 else 'left')

plt.axhline(-np.log10(0.05), c='grey', ls='--')
plt.axvline(1, c='grey', ls='--'); plt.axvline(-1, c='grey', ls='--')
plt.title("Volcano: PHA vs Unstim (Spatial Arcsinh)")
sns.despine()
plt.show()

# B. Heatmap
sc.pl.rank_genes_groups_heatmap(spatial_adata, n_genes=5, groupby='condition', 
                                standard_scale='var', cmap='RdBu_r', vmin=-2, vmax=2, 
                                figsize=(8, 8), swap_axes=True)

# 3. Top 20 Markers Table
top20_list = []
for group in spatial_adata.obs['condition'].unique():
    df = sc.get.rank_genes_groups_df(spatial_adata, group=group).head(20)
    df['group'] = group
    top20_list.append(df[['group', 'names', 'logfoldchanges', 'pvals_adj']])

final_df = pd.concat(top20_list).reset_index(drop=True)
print("Top 20 Spatial Features per Group:")
display(final_df)

In [ ]:
adata

In [ ]:
spatial_adata = ad.AnnData(X=adata.obsm['spatial_asinh5_top500_var'], obs=adata.obs)
spatial_adata.var_names = adata.obsm['spatial_asinh5_top500_var'].columns
sc.tl.rank_genes_groups(spatial_adata, groupby='condition', method='wilcoxon')

# 2. Figures
# A. Volcano Plot (PHA vs Unstim)
res = sc.get.rank_genes_groups_df(spatial_adata, group='PHA').copy()
res['nlog10'] = -np.log10(res['pvals_adj'] + 1e-300)
res['color'] = np.where((res.pvals_adj<.05) & (res.logfoldchanges>1), 'PHA', 
               np.where((res.pvals_adj<.05) & (res.logfoldchanges<-1), 'Unstim', 'NS'))

plt.figure(figsize=(10, 8))
sns.scatterplot(data=res, x='logfoldchanges', y='nlog10', hue='color', 
                palette={'PHA':'#d62728','Unstim':'#1f77b4','NS':'lightgrey'}, alpha=0.8)

# Add text labels for top 5 hits on each side
top_hits = pd.concat([res.nlargest(5, 'logfoldchanges'), res.nsmallest(5, 'logfoldchanges')])
for _, row in top_hits.iterrows():
    plt.text(row['logfoldchanges'], row['nlog10'], row['names'], fontsize=9, 
             ha='right' if row['logfoldchanges'] < 0 else 'left')

plt.axhline(-np.log10(0.05), c='grey', ls='--')
plt.axvline(1, c='grey', ls='--'); plt.axvline(-1, c='grey', ls='--')
plt.title("Volcano: PHA vs Unstim (Spatial Arcsinh)")
sns.despine()
plt.show()

# B. Heatmap
sc.pl.rank_genes_groups_heatmap(spatial_adata, n_genes=5, groupby='condition', 
                                standard_scale='var', cmap='RdBu_r', vmin=-2, vmax=2, 
                                figsize=(8, 8), swap_axes=True)

# 3. Top 20 Markers Table
top20_list = []
for group in spatial_adata.obs['condition'].unique():
    df = sc.get.rank_genes_groups_df(spatial_adata, group=group).head(20)
    df['group'] = group
    top20_list.append(df[['group', 'names', 'logfoldchanges', 'pvals_adj']])

final_df = pd.concat(top20_list).reset_index(drop=True)
print("Top 20 Spatial Features per Group:")
display(final_df)

In [ ]:
spatial_adata = ad.AnnData(X=adata.obsm['HOTSPOT_top500_var'], obs=adata.obs)
spatial_adata.var_names = adata.obsm['HOTSPOT_top500_var'].columns
sc.tl.rank_genes_groups(spatial_adata, groupby='condition', method='wilcoxon')

# 2. Figures
# A. Volcano Plot (PHA vs Unstim)
res = sc.get.rank_genes_groups_df(spatial_adata, group='PHA').copy()
res['nlog10'] = -np.log10(res['pvals_adj'] + 1e-300)
res['color'] = np.where((res.pvals_adj<.05) & (res.logfoldchanges>1), 'PHA', 
               np.where((res.pvals_adj<.05) & (res.logfoldchanges<-1), 'Unstim', 'NS'))

plt.figure(figsize=(10, 8))
sns.scatterplot(data=res, x='logfoldchanges', y='nlog10', hue='color', 
                palette={'PHA':'#d62728','Unstim':'#1f77b4','NS':'lightgrey'}, alpha=0.8)

# Add text labels for top 5 hits on each side
top_hits = pd.concat([res.nlargest(5, 'logfoldchanges'), res.nsmallest(5, 'logfoldchanges')])
for _, row in top_hits.iterrows():
    plt.text(row['logfoldchanges'], row['nlog10'], row['names'], fontsize=9, 
             ha='right' if row['logfoldchanges'] < 0 else 'left')

plt.axhline(-np.log10(0.05), c='grey', ls='--')
plt.axvline(1, c='grey', ls='--'); plt.axvline(-1, c='grey', ls='--')
plt.title("Volcano: PHA vs Unstim (Spatial Arcsinh)")
sns.despine()
plt.show()

# B. Heatmap
sc.pl.rank_genes_groups_heatmap(spatial_adata, n_genes=5, groupby='condition', 
                                standard_scale='var', cmap='RdBu_r', vmin=-2, vmax=2, 
                                figsize=(8, 8), swap_axes=True)

# 3. Top 20 Markers Table
top20_list = []
for group in spatial_adata.obs['condition'].unique():
    df = sc.get.rank_genes_groups_df(spatial_adata, group=group).head(20)
    df['group'] = group
    top20_list.append(df[['group', 'names', 'logfoldchanges', 'pvals_adj']])

final_df = pd.concat(top20_list).reset_index(drop=True)
print("Top 20 Spatial Features per Group:")
display(final_df)